Ranking de Pontualidade e Cancelamentos por Companhia Aérea

In [0]:
%sql
CREATE OR REPLACE TABLE gold_kpi_companhias AS
SELECT 
    nome_companhia AS Companhia_Aerea,
    COUNT(*) AS Total_Voos,
    SUM(CASE WHEN situacao_voo = 'CANCELADO' THEN 1 ELSE 0 END) AS Voos_Cancelados,
    ROUND(SUM(CASE WHEN situacao_voo = 'CANCELADO' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Taxa_Cancelamento_Percentual,
    ROUND(AVG(CASE WHEN situacao_voo = 'REALIZADO' THEN atraso_chegada_min END), 2) AS Media_Atraso_Minutos,
    ROUND(SUM(CASE WHEN status_pontualidade = 'Atrasado (>15 min)' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Taxa_Atraso_Percentual
FROM silver_voos_anac
GROUP BY nome_companhia
HAVING COUNT(*) > 1000
ORDER BY Total_Voos DESC;

SELECT * FROM gold_kpi_companhias;

Companhia_Aerea,Total_Voos,Voos_Cancelados,Taxa_Cancelamento_Percentual,Media_Atraso_Minutos,Taxa_Atraso_Percentual
LATAM Brasil (TAM),75033,1042,1.39,1.47,16.48
Azul Linhas Aéreas (AZU),70342,1061,1.51,2.36,14.76
GOL Linhas Aéreas (GLO),65712,706,1.07,-0.69,11.72
LATAM Airlines Chile (LAN),4276,39,0.91,4.91,16.67
Aerolíneas Argentinas (ARG),3235,98,3.03,13.91,28.66
Azul Conecta (ACN),3193,544,17.04,5.47,14.78
TAP Air Portugal (TAP),2555,23,0.90,4.16,25.71
Copa Airlines (CMP),2320,4,0.17,-0.9,12.07
JetSMART Argentina (JES),2068,66,3.19,9.61,13.93
Flybondi (FBZ),1712,142,8.29,33.64,30.20


Databricks visualization. Run in Databricks to view.

Top 10 Rotas Mais Críticas (Maior Média de Atraso)

In [0]:
%sql
CREATE OR REPLACE TABLE gold_rotas_criticas AS
SELECT 
    rota_descritiva AS Rota_Origem_Destino,
    COUNT(*) AS Quantidade_Voos,
    ROUND(AVG(atraso_chegada_min), 2) AS Media_Atraso_Minutos,
    ROUND(SUM(CASE WHEN status_pontualidade = 'Atrasado (>15 min)' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Taxa_Atraso_Percentual
FROM silver_voos_anac
WHERE situacao_voo = 'REALIZADO'
GROUP BY rota_descritiva
HAVING COUNT(*) > 200
ORDER BY Media_Atraso_Minutos DESC
LIMIT 10;

SELECT * FROM gold_rotas_criticas;

Rota_Origem_Destino,Quantidade_Voos,Media_Atraso_Minutos,Taxa_Atraso_Percentual
Miami (KMIA) -> Viracopos/Campinas (SBKP),268,78.92,38.81
Guarulhos (SBGR) -> Foz do Iguaçu (SBFI),639,77.06,19.25
Miami (KMIA) -> Guarulhos (SBGR),566,69.07,40.28
Guarulhos (SBGR) -> Buenos Aires/Ezeiza (SAEZ),389,30.7,37.79
Florianópolis (SBFL) -> Buenos Aires/Ezeiza (SAEZ),530,29.94,27.36
Buenos Aires/Aeroparque (SABE) -> Florianópolis (SBFL),565,27.85,35.22
Nova York/JFK (KJFK) -> Guarulhos (SBGR),261,27.8,34.10
Guarulhos (SBGR) -> Miami (KMIA),443,26.86,30.70
Viracopos/Campinas (SBKP) -> Vitória (SBVT),308,23.87,32.47
Córdoba (SACO) -> Rio de Janeiro/Galeão (SBGL),262,22.24,20.99


Databricks visualization. Run in Databricks to view.

Evolução Mensal de Voos e Pontualidade (1º Trimestre de 2026)

In [0]:
%sql
CREATE OR REPLACE TABLE gold_evolucao_mensal AS
SELECT 
    DATE_FORMAT(partida_prevista, 'yyyy-MM') AS mes_referencia,
    COUNT(*) AS total_voos,
    ROUND(AVG(CASE WHEN situacao_voo = 'REALIZADO' THEN atraso_chegada_min END), 2) AS media_atraso_minutos,
    ROUND(SUM(CASE WHEN situacao_voo = 'CANCELADO' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS taxa_cancelamento_percentual
FROM silver_voos_anac
WHERE partida_prevista IS NOT NULL
GROUP BY DATE_FORMAT(partida_prevista, 'yyyy-MM')
ORDER BY mes_referencia;

SELECT 
    mes_referencia AS Mes_Referencia,
    total_voos AS Total_Voos,
    media_atraso_minutos AS Media_Atraso_Minutos,
    taxa_cancelamento_percentual AS Taxa_Cancelamento_Percentual
FROM gold_evolucao_mensal;

Mes_Referencia,Total_Voos,Media_Atraso_Minutos,Taxa_Cancelamento_Percentual
2026-01,88942,4.7,3.16
2026-02,77134,4.21,3.08
2026-03,84642,2.22,3.10
2026-04,19,-87.07,21.05


Databricks visualization. Run in Databricks to view.